# 13_financial_calculations
All **internal-side metrics** are derived from:

- `silver.payment` — the `payment.csv` ledger
- `silver.commitment`

These internal metrics will later be compared against the **external-side metrics** from:
- `silver.cash_current`
- `silver.position_current`

The comparison will be performed during the **Reconciliation** stage.

---

## 1. Date Semantics Rule

The following date semantics have been decided and should be maintained unless explicitly revisited.

### External `.dat` Sources

For external `.dat` sources:

1. Use `settlement_timestamp` when it is present.
2. If `settlement_timestamp` is unavailable, fall back to `event_timestamp`.


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. Total Commitments
Straight from `commitment` - no filtering needed, every commitment
row is already a real, settled commitment (committing capital isn't
a payment event with a status).

In [0]:
commitment_df = spark.table(silver_table("commitment"))

total_commitments_df = (
    commitment_df
    .groupBy("fund_id")
    .agg(F.sum("committed_amount_usd").alias("total_commitments"))
)
total_commitments_df.show()

+--------+-----------------+
| fund_id|total_commitments|
+--------+-----------------+
|FUND_001|     1.93012514E8|
|FUND_002|      8.1854831E7|
|FUND_003|     1.40148036E8|
|FUND_004|     1.91155002E8|
|FUND_005|     1.47994475E8|
+--------+-----------------+



## 2. Contributions, Capital Calls, Invested Capital (internal, from payment.csv)
All three follow the same shape: sum settled amount_usd per fund,
filtered to one payment_type.

In [0]:
payment_df = spark.table(silver_table("payment"))

def settled_sum_by_type(payment_type: str, out_col: str):
    return (
        payment_df
        .filter((F.col("payment_type") == payment_type) & (F.col("status") == "SETTLED"))
        .groupBy("fund_id")
        .agg(F.sum("amount_usd").alias(out_col))
    )

contributions_df = settled_sum_by_type("CONTRIBUTION", "contributions")
capital_calls_df = settled_sum_by_type("CAPITAL_CALL", "capital_calls_settled")
invested_capital_df = settled_sum_by_type("INVESTMENT", "invested_capital")

contributions_df.show()
capital_calls_df.show()
invested_capital_df.show()

+--------+--------------------+
| fund_id|       contributions|
+--------+--------------------+
|FUND_001|3.3175863380000003E7|
|FUND_002|1.3992916270000003E7|
|FUND_004|       2.485377029E7|
|FUND_003|2.1703094939999998E7|
|FUND_005|          8944997.92|
+--------+--------------------+

+--------+---------------------+
| fund_id|capital_calls_settled|
+--------+---------------------+
|FUND_005|        2.564814512E7|
|FUND_003|        2.453312945E7|
|FUND_004|        2.662715298E7|
|FUND_002|           1435030.71|
|FUND_001| 2.5525862150000002E7|
+--------+---------------------+

+--------+------------------+
| fund_id|  invested_capital|
+--------+------------------+
|FUND_004|2139281.6999999997|
|FUND_001|        7814463.41|
|FUND_003|         704845.78|
|FUND_002|2533459.2899999996|
+--------+------------------+



## 3. Uncalled Capital
Total Commitments minus settled Capital Calls. Funds with zero
settled capital calls still need a row (COALESCE to 0), so this is a
left join, not an inner join.

In [0]:
uncalled_capital_df = (
    total_commitments_df
    .join(capital_calls_df, on="fund_id", how="left")
    .withColumn("capital_calls_settled", F.coalesce(F.col("capital_calls_settled"), F.lit(0.0)))
    .withColumn("uncalled_capital", F.col("total_commitments") - F.col("capital_calls_settled"))
    .select("fund_id", "uncalled_capital")
)
uncalled_capital_df.show()

+--------+----------------+
| fund_id|uncalled_capital|
+--------+----------------+
|FUND_004|  1.6452784902E8|
|FUND_002|   8.041980029E7|
|FUND_003|  1.1561490655E8|
|FUND_005|  1.2234632988E8|
|FUND_001|  1.6748665185E8|
+--------+----------------+



## 4. Portfolio Value
Quantity held (latest business_date per fund+asset, from
`position_current`) x latest `close` price (from `market_price`),
joined via `portfolio_company` to resolve `benchmark_ticker`. NOTE:
an earlier version of this cell summed price alone without quantity


In [0]:
portfolio_company_df = spark.table(silver_table("portfolio_company"))
market_price_df = spark.table(silver_table("market_price"))
position_current_df = spark.table(silver_table("position_current"))

w_price = Window.partitionBy("ticker").orderBy(F.col("bar_timestamp").desc())
latest_price_df = (
    market_price_df
    .withColumn("_rn", F.row_number().over(w_price))
    .filter(F.col("_rn") == 1)
    .select("ticker", F.col("close").alias("latest_close"))
)

w_qty = Window.partitionBy("internal_fund_id", "internal_asset_id").orderBy(F.col("business_date").desc())
latest_position_df = (
    position_current_df
    .withColumn("_rn", F.row_number().over(w_qty))
    .filter(F.col("_rn") == 1)
    .select("internal_fund_id", "internal_asset_id", "quantity")
)

company_valuation_df = (
    portfolio_company_df
    .join(latest_price_df, portfolio_company_df["benchmark_ticker"] == latest_price_df["ticker"], how="left")
    .join(
        latest_position_df,
        (portfolio_company_df["fund_id"] == latest_position_df["internal_fund_id"]) &
        (portfolio_company_df["company_id"] == latest_position_df["internal_asset_id"]),
        how="left"
    )
    .withColumn("has_benchmark", F.col("latest_close").isNotNull())
    .withColumn("has_position", F.col("quantity").isNotNull())
    .withColumn(
        "market_value",
        F.when(F.col("has_benchmark") & F.col("has_position"), F.col("latest_close") * F.col("quantity"))
    )
    .select("company_id", "company_name", "fund_id", "benchmark_ticker", "latest_close", "quantity", "market_value", "has_benchmark", "has_position")
)

no_benchmark_count = company_valuation_df.filter(~F.col("has_benchmark")).count()
no_position_count = company_valuation_df.filter(~F.col("has_position")).count()
print(f"Companies with no benchmark price: {no_benchmark_count}")
print(f"Companies with no position data (yet): {no_position_count}")
company_valuation_df.show(truncate=False)

portfolio_value_df = (
    company_valuation_df
    .filter(F.col("market_value").isNotNull())
    .groupBy("fund_id")
    .agg(F.sum("market_value").alias("portfolio_value"))
)
portfolio_value_df.show()

Companies with no benchmark price: 6
Companies with no position data (yet): 29
+----------+-----------------------------+--------+----------------+------------+--------+------------+-------------+------------+
|company_id|company_name                 |fund_id |benchmark_ticker|latest_close|quantity|market_value|has_benchmark|has_position|
+----------+-----------------------------+--------+----------------+------------+--------+------------+-------------+------------+
|PORT_0001 |Henson-Johnston              |FUND_005|UNH             |377.54      |NULL    |NULL        |true         |false       |
|PORT_0002 |Ramos, Carr and Cook         |FUND_002|OR.PA           |383.65      |NULL    |NULL        |true         |false       |
|PORT_0003 |Lang-Anderson                |FUND_001|MC.PA           |406.9       |2580.0  |1049802.0   |true         |true        |
|PORT_0004 |Lee LLC                      |FUND_003|ULVR.L          |4648.5      |NULL    |NULL        |true         |false       |
|POR

## 5. Available Cash - Internal
Settled Contributions - Invested Capital.

In [0]:
available_cash_internal_df = (
    total_commitments_df.select("fund_id")
    .join(contributions_df, on="fund_id", how="left")
    .join(invested_capital_df, on="fund_id", how="left")
    .fillna(0.0, subset=["contributions", "invested_capital"])
    .withColumn(
        "available_cash_internal",
        F.col("contributions") - F.col("invested_capital")
    )
    .select("fund_id", "available_cash_internal")
)
available_cash_internal_df.show()

+--------+-----------------------+
| fund_id|available_cash_internal|
+--------+-----------------------+
|FUND_004|          2.271448859E7|
|FUND_002|   1.1459456980000004E7|
|FUND_003|   2.0998249159999996E7|
|FUND_005|             8944997.92|
|FUND_001|   2.5361399970000003E7|
+--------+-----------------------+



## 6. Available Cash - External
Straight from the bank-side feed (`cash_current`, settled only) -
the ground-truth balance Reconciliation will compare Internal
against.

In [0]:
cash_current_df = spark.table(silver_table("cash_current"))

daily_balance_df = (
    cash_current_df
    .filter(F.col("status") == "SETTLED")
    .groupBy("internal_fund_id", "business_date")
    .agg(F.sum("amount").alias("daily_balance"))
)

w_latest_balance = Window.partitionBy("internal_fund_id").orderBy(F.col("business_date").desc())
available_cash_external_df = (
    daily_balance_df
    .withColumn("_rn", F.row_number().over(w_latest_balance))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("internal_fund_id").alias("fund_id"),
        F.col("daily_balance").alias("available_cash_external")
    )
)
available_cash_external_df.show()

+--------+-----------------------+
| fund_id|available_cash_external|
+--------+-----------------------+
|FUND_001|                 1.31E7|
|FUND_002|              8450000.0|
|FUND_003|              5580000.0|
+--------+-----------------------+



## 7. Estimated NAV
Portfolio Value + Available Cash (Internal). Uses the internal cash
figure as the fund's official position - external is the
reconciliation comparison point, not a second NAV input. Documented
simplification for this POC, not a full fund-accounting NAV.

In [0]:
nav_df = (
    total_commitments_df.select("fund_id")
    .join(portfolio_value_df, on="fund_id", how="left")
    .join(available_cash_internal_df, on="fund_id", how="left")
    .fillna(0.0, subset=["portfolio_value", "available_cash_internal"])
    .withColumn("estimated_nav", F.col("portfolio_value") + F.col("available_cash_internal"))
    .select("fund_id", "estimated_nav")
)
nav_df.show()

+--------+--------------------+
| fund_id|       estimated_nav|
+--------+--------------------+
|FUND_004|       2.271448859E7|
|FUND_002|1.1459456980000004E7|
|FUND_003|2.0998249159999996E7|
|FUND_005|          8944997.92|
|FUND_001|2.6411201970000003E7|
+--------+--------------------+



## 8. Assemble silver.fund_financials
One row per fund, every metric above joined together - this is what
Gold_Fund_Financials will read directly.

In [0]:
fund_financials_df = (
    total_commitments_df
    .join(contributions_df, on="fund_id", how="left")
    .join(uncalled_capital_df, on="fund_id", how="left")
    .join(invested_capital_df, on="fund_id", how="left")
    .join(portfolio_value_df, on="fund_id", how="left")
    .join(available_cash_internal_df, on="fund_id", how="left")
    .join(available_cash_external_df, on="fund_id", how="left")
    .join(nav_df, on="fund_id", how="left")
    .fillna(0.0, subset=[
        "contributions", "uncalled_capital", "invested_capital",
        "portfolio_value", "available_cash_internal", "available_cash_external", "estimated_nav"
    ])
    .withColumn("calculated_at", F.current_timestamp())
)

fund_financials_df.show(truncate=False)

+--------+-----------------+--------------------+----------------+------------------+---------------+-----------------------+-----------------------+--------------------+--------------------------+
|fund_id |total_commitments|contributions       |uncalled_capital|invested_capital  |portfolio_value|available_cash_internal|available_cash_external|estimated_nav       |calculated_at             |
+--------+-----------------+--------------------+----------------+------------------+---------------+-----------------------+-----------------------+--------------------+--------------------------+
|FUND_004|1.91155002E8     |2.485377029E7       |1.6452784902E8  |2139281.6999999997|0.0            |2.271448859E7          |0.0                    |2.271448859E7       |2026-09-23 03:34:08.153458|
|FUND_002|8.1854831E7      |1.3992916270000003E7|8.041980029E7   |2533459.2899999996|0.0            |1.1459456980000004E7   |8450000.0              |1.1459456980000004E7|2026-09-23 03:34:08.153458|
|FUND_003|

In [0]:
write_silver(fund_financials_df, "fund_financials")
print(f"silver.fund_financials row count: {fund_financials_df.count()}")

silver.fund_financials row count: 5


## 9. Also persist company-level valuation
For Gold_Portfolio_Valuation  - company grain, not fund grain.

In [0]:
write_silver(company_valuation_df, "portfolio_valuation")
print(f"silver.portfolio_valuation row count: {company_valuation_df.count()}")

silver.portfolio_valuation row count: 30


### Summary
`silver.fund_financials` (fund grain) and `silver.portfolio_valuation`
(company grain) are now ready for Gold  and for Reconciliation
to compare `available_cash_internal` against `cash_current`
(external) directly.

In [0]:
%sql

SELECT payment_id, commitment_id, fund_id, payment_type, amount_usd, status, event_date
FROM dbw_pe_platform.silver.payment
WHERE fund_id = 'FUND_004' AND payment_type IN ('CAPITAL_CALL', 'CONTRIBUTION')
ORDER BY commitment_id, event_date;

payment_id,commitment_id,fund_id,payment_type,amount_usd,status,event_date
PMT_000117,CMT_00038,FUND_004,CAPITAL_CALL,2981129.88,INITIATED,2026-09-19
PMT_000119,CMT_00038,FUND_004,CAPITAL_CALL,1275222.54,SETTLED,2026-09-19
PMT_000115,CMT_00038,FUND_004,CAPITAL_CALL,3317998.23,SETTLED,2026-09-19
PMT_000163,CMT_00038,FUND_004,CONTRIBUTION,890791.07,SETTLED,2026-09-20
PMT_000130,CMT_00038,FUND_004,CONTRIBUTION,2350723.13,SETTLED,2026-09-20
PMT_000020,CMT_00039,FUND_004,CONTRIBUTION,2922636.63,FAILED,2026-09-17
PMT_000147,CMT_00039,FUND_004,CONTRIBUTION,4031078.3,INITIATED,2026-09-20
PMT_000208,CMT_00039,FUND_004,CAPITAL_CALL,2598210.22,SETTLED,2026-09-21
PMT_000087,CMT_00041,FUND_004,CONTRIBUTION,2960945.39,SETTLED,2026-09-19
PMT_000143,CMT_00041,FUND_004,CONTRIBUTION,2465509.54,SETTLED,2026-09-20


In [0]:
%sql
SELECT
  c.fund_id,
  SUM(c.contributed_total) AS commitment_reported_contributed_total,
  SUM(CASE WHEN p.payment_type = 'CONTRIBUTION' AND p.status = 'SETTLED' THEN p.amount_usd ELSE 0 END) AS our_computed_contributions,
  SUM(CASE WHEN p.payment_type = 'CAPITAL_CALL' AND p.status = 'SETTLED' THEN p.amount_usd ELSE 0 END) AS our_computed_capital_calls
FROM dbw_pe_platform.silver.commitment c
LEFT JOIN dbw_pe_platform.silver.payment p ON c.fund_id = p.fund_id
GROUP BY c.fund_id
ORDER BY c.fund_id;


fund_id,commitment_reported_contributed_total,our_computed_contributions,our_computed_capital_calls
FUND_001,0.0,5.63989677459999E8,4.3393965654999954E8
FUND_002,0.0,1.1194333016000009E8,1.1480245679999998E7
FUND_003,0.0,2.604371392799999E8,2.943975533999999E8
FUND_004,0.0,4.225140949300002E8,4.526616006599992E8
FUND_005,0.0,1.0733997503999998E8,3.077777414399998E8
